In [ ]:
import os
import datasets as hfds

from olmo import Tokenizer
from olmo_data import get_data_path


from zsl_config import ZSL_DIR_DATA

SEQ_LEN = 1024

In [ ]:
paloma_token = "your_token_here" # agree to T&C here https://huggingface.co/datasets/allenai/paloma
c4_en_val = hfds.load_dataset("allenai/paloma", "c4_en", split="val", token=paloma_token)

In [ ]:
def load_olmo_tokenizer(tokenizer_identifier: str = 'tokenizers/allenai_eleuther-ai-gpt-neox-20b-pii-special.json', eos_token_id: int = 0, pad_token_id: int = 1):
	with get_data_path(tokenizer_identifier) as tokenizer_path:
		tokenizer = Tokenizer.from_file(
			tokenizer_path,
			eos_token_id=eos_token_id,
			pad_token_id=pad_token_id,
		)
	return tokenizer

tokenizer_olmo = load_olmo_tokenizer()

def tokenize_olmo(element):
    outputs = tokenizer_olmo.encode_batch(
        element["text"],
    )
    input_batch = []
    seq = []
    for input_ids in outputs:
        seq += input_ids
        seq_len = len(seq)
        if seq_len <= SEQ_LEN:
            continue
        else:
            seq_len = SEQ_LEN * (seq_len // SEQ_LEN)
            input_batch += [seq[i:i+SEQ_LEN] for i in range(0, seq_len, SEQ_LEN)]
            seq = seq[seq_len:]

    return {"input_ids": input_batch}

tokenized_dir = ZSL_DIR_DATA / "tokenized/olmo-c4_en_val"
tokenized = c4_en_val.map(tokenize_olmo, batched=True, remove_columns=c4_en_val.column_names)
tokenized = tokenized.shuffle(seed=42)
tokenized.save_to_disk(tokenized_dir)

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")

def tokenize_nanogpt(element):
    outputs = [enc.encode_ordinary(x) + [enc.eot_token] for x in element["text"]]
    input_batch = []
    seq = []
    for input_ids in outputs:
        seq += input_ids
        seq_len = len(seq)
        if seq_len <= SEQ_LEN:
            continue
        else:
            seq_len = SEQ_LEN * (seq_len // SEQ_LEN)
            input_batch += [seq[i:i+SEQ_LEN] for i in range(0, seq_len, SEQ_LEN)]
            seq = seq[seq_len:]

    return {"input_ids": input_batch} 

tokenized_dir = ZSL_DIR_DATA / "tokenized/nanogpt-c4_en_val"
tokenized = c4_en_val.map(tokenize_nanogpt, batched=True, remove_columns=c4_en_val.column_names)
tokenized = tokenized.shuffle(seed=42)
tokenized.save_to_disk(tokenized_dir)